# Buổi 5 — Python cho dữ liệu PISA: pipeline, so sánh nhiều nước, chạm ML
Phần A chạy trên Colab (dữ liệu VN nhúng). Phần B (721.037 học sinh, 84 nước) **chỉ chạy được trên máy có file gốc** — ô sẽ tự bỏ qua nếu không thấy file.

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

> 📥 **Đầu vào:** nạp thư viện.

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


> 📥 **Đầu vào:** tải lại 2 bảng PISA VN.
>
> 📤 **Đầu ra thật:** `(195, 36)` và `(7368, 54)`. ✅

## A1. Pipeline có thể lặp lại

In [3]:
def pipeline(t,h):
    t=t.copy(); t["tu"]=(t.PRIVATESCH==2).astype(int); t["cao"]=(t.sci_mean>t.sci_mean.median()).astype(int); return t
T=pipeline(truong,hs)
T.groupby("vung_ten").agg(n=("CNTSCHID","count"),sci=("sci_mean","mean"),edulead=("EDULEAD","mean")).round(3)

,n,sci,edulead
vung_ten,,,
"Bắc TB, DH TB & Tây Nguyên",58,0.426,0.895
Trung du & MN phía Bắc,29,0.390,0.723
ĐB sông Cửu Long,37,0.438,0.381
ĐB sông Hồng,43,0.493,0.949
Đông Nam Bộ,28,0.483,0.766


> 📥 **Đầu vào:** hàm `pipeline()` đóng gói 2 bước xử lý (tạo biến `tu`, tạo biến nhị phân `cao` = 1 nếu trường có `sci_mean` trên TRUNG VỊ) thành MỘT hàm có thể GỌI LẠI NHIỀU LẦN — thực hành kỹ thuật lập trình tốt (viết pipeline thay vì lặp code thủ công), tương ứng bước 4 "Làm sạch dữ liệu" trong sơ đồ pipeline tổng thể của site.
>
> 📤 **Đầu ra thật:** bảng tổng hợp theo vùng — khớp CHÍNH XÁC với notebook buổi 3 (vd ĐB sông Hồng sci=0,493, EDULEAD=0,949 — cao nhất cả về điểm số VÀ lãnh đạo giáo dục; Cửu Long sci=0,438 nhưng EDULEAD chỉ 0,381 — thấp nhất). ✅ Xác nhận pipeline hoạt động đúng, cho kết quả nhất quán với các notebook trước.

## A2. Học máy trên n=195: dự đoán trường thuộc nửa trên về điểm thô Khoa học
Đánh giá bằng AUC qua kiểm chứng chéo 5 lần (0.5 = đoán mò).

In [4]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
for i in [2,3,4,5]: T[f"v{i}"]=(T.vung==i).astype(int)
y=T.cao; cv=StratifiedKFold(5,shuffle=True,random_state=0)
sets={"chỉ 7 chỉ số WLE":W,"chỉ vùng":["v2","v3","v4","v5"],"WLE + loại + vùng":W+["tu","v2","v3","v4","v5"]}
for n,c in sets.items(): print(f"Logistic · {n}: AUC={cross_val_score(LogisticRegression(max_iter=2000),T[c],y,cv=cv,scoring='roc_auc').mean():.2f}")
print("Random Forest · WLE + loại + vùng: AUC=%.2f"%cross_val_score(RandomForestClassifier(300,random_state=0),T[sets["WLE + loại + vùng"]],y,cv=cv,scoring="roc_auc").mean())

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:219: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights_xp + intercept_xp
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:219: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights_xp + intercept_xp
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:219: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights_xp + intercept_xp
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:219: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights_xp + intercept_xp
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:219: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights_xp + intercept_xp
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_linear_

Logistic · chỉ 7 chỉ số WLE: AUC=0.54
Logistic · chỉ vùng: AUC=0.64
Logistic · WLE + loại + vùng: AUC=0.67


Random Forest · WLE + loại + vùng: AUC=0.58


> 📥 **Đầu vào:** bài toán học máy (ML) THẬT: dự đoán biến nhị phân `cao` (trường có điểm Khoa học trên trung vị hay không) từ 3 bộ biến khác nhau, đánh giá bằng AUC (Area Under Curve — 0,5 = đoán ngẫu nhiên, 1,0 = hoàn hảo) qua 5-fold cross-validation (chia dữ liệu 5 phần, luân phiên train/test để đánh giá khách quan hơn).
>
> ⚠️ **Về các dòng `RuntimeWarning`:** đây là cảnh báo THẬT từ scikit-learn khi hồi quy logistic gặp khó khăn hội tụ ở một vài fold (cỡ mẫu 195 chia 5 fold chỉ còn ~156 dòng train, khá nhỏ cho logistic regression với nhiều biến) — không phải lỗi nghiêm trọng, AUC vẫn tính ra được bình thường, nhưng là dấu hiệu thực tế rằng **n=195 khá nhỏ cho bài toán ML có nhiều biến**.
>
> 📤 **Đầu ra thật:** `chỉ 7 chỉ số WLE: AUC=0,54` (gần như ngẫu nhiên — khớp đúng phát hiện ở notebook buổi 4 rằng các chỉ số WLE gần như không dự đoán được điểm số!); `chỉ vùng: AUC=0,64` (tốt hơn hẳn dù chỉ dùng 1 biến định danh); `WLE + loại + vùng: AUC=0,67` (tốt nhất, nhưng KHÔNG cải thiện nhiều so với chỉ dùng vùng — 0,64→0,67); Random Forest (mô hình phức tạp hơn) chỉ đạt AUC=0,58, THẤP HƠN cả logistic đơn giản với vùng!
>
> 🎯 **Trả lời câu hỏi ❓ bên dưới:** vùng miền dự đoán tốt hơn 7 chỉ số trường học vì (đã xác nhận ở buổi 3-4) chênh lệch điểm số THẬT SỰ chủ yếu nằm ở cấp vùng miền (kinh tế-xã hội), không phải ở các chỉ số môi trường trường học đo được. Với n=195 và AUC≈0,6-0,7, đây là mức dự đoán **KHIÊM TỐN** — không nên dùng mô hình này để "chấm điểm"/xếp loại từng trường cụ thể trong thực tế, chỉ phù hợp để tìm HƯỚNG khác biệt tổng quát giữa các nhóm trường. Việc Random Forest (mô hình mạnh hơn về lý thuyết) lại cho AUC THẤP HƠN logistic đơn giản là bài học ML quan trọng: với cỡ mẫu NHỎ (195), mô hình phức tạp dễ bị OVERFIT (học "nhiễu" thay vì mẫu hình thật), mô hình đơn giản thường đáng tin cậy hơn.

**❓** Vì sao thông tin *vùng* dự đoán tốt hơn 7 chỉ số trường học? Với n = 195 và AUC ≈ 0,6–0,7, ta nên kết luận gì?

## B. So sánh nhiều nước (chỉ chạy được khi có file gốc)

In [5]:
import os
P="/Users/tuannghiat/Downloads/Đào tạo riêng - SPSS for Social Science/PISA 2025/_giai_nen/CY09_MS_STU_TT_PUF.sav"
if os.path.exists(P):
    import pyreadstat
    tt,_=pyreadstat.read_sav(P,encoding="latin1")            # STU_TT cần encoding latin1 với pyreadstat
    cols=[c for c in tt.columns if c.endswith("_TT") and c!="EFFORT_TT"]; tt["qq_time"]=tt[cols].sum(axis=1,min_count=1)
    cm=tt.groupby("CNT").qq_time.median().sort_values(); print("VN:",cm["VNM"],"| hạng",int((cm<cm["VNM"]).sum()+1),"/",len(cm),"| trung vị toàn bộ:",tt.qq_time.median())
    print({k:round(cm[k]) for k in ["VNM","THA","IDN","MYS","PHL","SGP","BRN"] if k in cm.index})
else: print("Không thấy file gốc → bỏ qua phần B (bình thường trên Colab).")

VN: 1805.0 | hạng 43 / 84 | trung vị toàn bộ: 1802.0
{'VNM': 1805, 'THA': 1696, 'IDN': 2122, 'MYS': 2694, 'PHL': 1839, 'SGP': 1641, 'BRN': 2430}


> 📥 **Đầu vào:** file gốc PISA đầy đủ TOÀN CẦU (`CY09_MS_STU_TT_PUF.sav`, chứa dữ liệu 84 quốc gia/vùng lãnh thổ tham gia PISA — file này CHỈ có trên máy cá nhân người phụ trách khoá học, không public, nên đoạn code có `if os.path.exists()` để tự động BỎ QUA an toàn nếu chạy trên Colab/máy khác không có file này).
>
> 📤 **Đầu ra thật (chạy được trên máy nguồn):** thời gian làm bài trung vị của Việt Nam = **1.805 giây**, xếp hạng **43/84** quốc gia (đứng giữa bảng xếp hạng toàn cầu về thời gian làm bài — không nhanh không chậm bất thường). So sánh khu vực Đông Nam Á: Thái Lan **1.696s** (nhanh hơn VN), Indonesia **2.122s**, Malaysia **2.694s** (chậm nhất khu vực), Philippines **1.839s** (gần VN), Singapore **1.641s** (nhanh nhất khu vực — hợp lý với hệ thống giáo dục có tiếng làm bài thi hiệu quả), Brunei **2.430s**.
>
> 🎯 **Ý nghĩa:** thời gian làm bài trung vị KHÔNG hẳn phản ánh "học sinh giỏi hay kém" — có thể phản ánh sự khác biệt về chiến lược làm bài, tốc độ đọc hiểu ngôn ngữ bản địa, hoặc cách tổ chức thi. Việt Nam ở vị trí trung bình thế giới (hạng 43/84) là một điểm dữ liệu tham khảo thú vị nhưng cần thận trọng khi diễn giải nguyên nhân.

## Bài tập
`bai_tap/buoi5_de.md`